# Adaptive V2 Scale Campaign
CPU-only, frozen 300-run campaign. Run only after uploading `kaggle_upload_adaptive_scale.zip`.

In [ ]:
from pathlib import Path
import os, shutil, subprocess, zipfile
root = Path('/kaggle/working/SIGMOD')
if root.exists(): shutil.rmtree(root)
bundles = list(Path('/kaggle/input').rglob('kaggle_upload_adaptive_scale.zip'))
if len(bundles) == 1:
    root.mkdir(parents=True)
    with zipfile.ZipFile(bundles[0]) as archive: archive.extractall(root)
else:
    sources = [p.parent for p in Path('/kaggle/input').rglob('campaign_git_commit.txt')]
    assert not bundles and len(sources) == 1, f'Expected one ZIP or extracted campaign dataset; ZIPs={len(bundles)}, sources={len(sources)}'
    shutil.copytree(sources[0], root)
required = [root/'experiments/configs/scale.json', root/'experiments/campaign.py', root/'scripts/linux/run_scale.sh', root/'campaign_git_commit.txt']
assert all(p.is_file() for p in required), [str(p) for p in required if not p.is_file()]
commit = (root/'campaign_git_commit.txt').read_text().strip()
assert len(commit) >= 7 and all(c in '0123456789abcdefABCDEF' for c in commit), commit
os.environ['CABTREE_GIT_COMMIT'] = commit
os.environ['CABTREE_CXXFLAGS'] = '-std=c++11 -O3 -march=native -DNDEBUG'
print('source commit:', commit)
print('root:', root)


In [ ]:
# CPU identity only. Accelerator must be None; this cell does not change other sessions.
subprocess.run(['uname', '-a'], check=True)
subprocess.run(['lscpu'], check=True)
subprocess.run(['free', '-h'], check=True)
subprocess.run(['g++', '--version'], check=True)


In [ ]:
# Build the committed source and run the existing semantic V2 unit gate before performance.
subprocess.run(['bash', str(root/'scripts/linux/build_release.sh'), '--execute'], cwd=root, check=True, env=os.environ.copy())
subprocess.run([str(root/'build/prepared/test_v2')], cwd=root, check=True, env=os.environ.copy())


In [ ]:
# Fail closed if the frozen matrix no longer expands to exactly 300 rows.
check = subprocess.run(['bash', str(root/'scripts/linux/run_scale.sh'), 'dry-run', 'kaggle'], cwd=root, check=True, capture_output=True, text=True, env=os.environ.copy())
assert 'planned runs: 300' in check.stdout, check.stdout[-2000:]
print('Frozen dry-run count: 300')


In [ ]:
# Primary execution cell. Safe resume skips atomically completed rows.
subprocess.run(['bash', str(root/'scripts/linux/run_scale.sh'), '--execute', 'kaggle'], cwd=root, check=True, env=os.environ.copy())


In [ ]:
# Structural completion check; scientific pairing is performed later by analysis/adaptive_scale.py.
raw = root/'results/raw/kaggle'
main = [p for p in raw.rglob('*.csv') if not p.name.endswith('.phases.csv') and '.partial' not in p.name]
phases = list(raw.rglob('*.phases.csv'))
partials = list(raw.rglob('*.partial.csv'))
failures = list(raw.rglob('*.failed.json'))
print('main', len(main), 'phase', len(phases), 'partial', len(partials), 'failed', len(failures))
assert len(main) == 300 and len(phases) == 300 and not partials and not failures


In [ ]:
# Final cell: downloadable evidence plus a resume bundle.
results_zip = shutil.make_archive('/kaggle/working/adaptive_scale_results', 'zip', root/'results')
resume_root = Path('/kaggle/working/adaptive_scale_resume_bundle')
if resume_root.exists(): shutil.rmtree(resume_root)
shutil.copytree(root/'results', resume_root/'results')
shutil.copy2(root/'experiments/configs/scale.json', resume_root/'scale.json')
shutil.copy2(root/'campaign_git_commit.txt', resume_root/'campaign_git_commit.txt')
resume_zip = shutil.make_archive('/kaggle/working/adaptive_scale_resume', 'zip', resume_root)
print(results_zip)
print(resume_zip)
